# NYC Film Permits — Most Filmed Blocks
This notebook parses the `ParkingHeld` column from the NYC film permits dataset to find which city blocks appear most frequently.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# Load the data
df = pd.read_excel('Film_Permits_20260416.xlsx')
print(f'Total permits: {len(df):,}')
print(f'Date range: {df["StartDateTime"].min()[:10]} to {df["StartDateTime"].max()[:10]}')
df.head(3)

In [ ]:
# Preview the raw ParkingHeld column
print('Example ParkingHeld values:\n')
for val in df['ParkingHeld'].dropna().head(3):
    print(repr(val))
    print()

In [ ]:
def parse_blocks(parking_held_value):
    """
    Split a ParkingHeld cell into individual block descriptions.
    Each block looks like: 'STREET A between CROSS_ST_1 and CROSS_ST_2'
    Blocks are comma-separated within each cell.
    Returns a list of cleaned block strings.
    """
    if pd.isna(parking_held_value):
        return []
    
    # Split on commas, strip whitespace, drop empty strings
    blocks = [b.strip() for b in str(parking_held_value).split(',')]
    blocks = [b for b in blocks if len(b) > 3]  # filter out stray commas/blanks
    
    # Normalize to uppercase for consistent counting
    blocks = [b.upper() for b in blocks]
    
    return blocks

# Test on first row
sample = df['ParkingHeld'].dropna().iloc[0]
print('Raw:', sample)
print()
print('Parsed blocks:')
for b in parse_blocks(sample):
    print(' -', b)

In [ ]:
# Apply parser to entire dataset and count block appearances
all_blocks = []
for val in df['ParkingHeld']:
    all_blocks.extend(parse_blocks(val))

block_counts = Counter(all_blocks)

print(f'Total block-permit appearances: {len(all_blocks):,}')
print(f'Unique blocks: {len(block_counts):,}')

In [ ]:
# Top 20 most filmed blocks
top_blocks = pd.DataFrame(
    block_counts.most_common(20),
    columns=['Block', 'Permit Count']
)
top_blocks.index = top_blocks.index + 1  # start ranking at 1
top_blocks

In [ ]:
# Bar chart of top 20 blocks
fig, ax = plt.subplots(figsize=(10, 7))

top20 = top_blocks.head(20)

# Shorten labels for readability
labels = [b[:55] + '...' if len(b) > 55 else b for b in top20['Block']]

bars = ax.barh(labels[::-1], top20['Permit Count'][::-1], color='steelblue', edgecolor='white')

# Add count labels
for bar, count in zip(bars, top20['Permit Count'][::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            str(count), va='center', fontsize=9)

ax.set_xlabel('Number of Filming Permits')
ax.set_title('Top 20 Most Filmed Blocks in NYC\n(by ParkingHeld permit appearances)', fontsize=13)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('top_blocks.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as top_blocks.png')

In [ ]:
# Bonus: breakdown by borough
# Explode ParkingHeld so each block gets its own row, then merge borough info

exploded_rows = []
for _, row in df.iterrows():
    for block in parse_blocks(row['ParkingHeld']):
        exploded_rows.append({'Block': block, 'Borough': row['Borough'], 'Category': row['Category']})

df_blocks = pd.DataFrame(exploded_rows)

borough_counts = df_blocks.groupby('Borough')['Block'].count().sort_values(ascending=False)
print('Total block-permit appearances by borough:')
print(borough_counts.to_string())

In [ ]:
# Top 10 blocks per borough
for borough in borough_counts.index:
    subset = df_blocks[df_blocks['Borough'] == borough]
    top = subset['Block'].value_counts().head(5)
    print(f'\n--- {borough} ---')
    for block, count in top.items():
        print(f'  {count:>4}x  {block}')